# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Run this to get the notebook working on DSMLP

In [1]:
import os

# Define the new path you want to add
user = os.environ.get("USER")
new_path = "/home/" + user + "/.local/bin"

# Append it to the existing PATH
os.environ['PATH'] += f":{new_path}"

# Verify the update
print(os.environ['PATH'])

/opt/k8s-support/bin:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/acojocaru/.local/bin


### Comment Out the cell below after first installation.

In [8]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install accelerate sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

downloading uv 0.11.12 x86_64-unknown-linux-gnu


installing to /home/acojocaru/.local/bin


  uv


  uvx


everything's installed!


...

(EngineCore pid=1967) INFO 05-10 11:06:48 [core.py:1238] Shutdown initiated (timeout=0)
(EngineCore pid=1967) INFO 05-10 11:06:48 [core.py:1261] Shutdown complete


/home/acojocaru/151B_SP26_Competition/.venv/bin/python: No module named pip


[rank0]:[W510 11:06:49.876803826 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Installed kernelspec cse151b in /home/acojocaru/.local/share/jupyter/kernels/cse151b


Done. Restart the kernel before proceeding.
Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.


### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
# Only need if not using quantization
# os.environ["VLLM_USE_DEEP_GEMM"] = "0"

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())     # Returns True if CUDA is available
if torch.cuda.is_available():
    current_device = torch.cuda.current_device()
    print("Current device ID:", current_device)
    print("Current device name:", torch.cuda.get_device_name(current_device))
    print("Device memory address:", torch.cuda.device(current_device))
    print("Total number of GPUs:", torch.cuda.device_count())

CUDA available: True
Current device ID: 0
Current device name: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition MIG 1g.24gb
Device memory address: <torch.cuda.device object at 0x7f2e2f748440>
Total number of GPUs: 1


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$pi*sqrt{a}$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "G",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [81]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "DO NOT FORMAT YOUR ANSWER WITH LATEX INSIDE THE BOXED PART. " 
    "IMPORTANT: **Ignore the question's expectation of the format of the answer**. Do not put any decimals in your answers EXCEPT those provided by the question. Always express numerical answers in !!EXACT!! form as ASCII Math:\n"
    "- Use square roots instead of decimals (e.g. sqrt(2) not 1.414)\n" 
    "- Use pi instead of 3.14159\n" 
    "- You may use trig functions and other expressions such as cos() or atan() AS LONG AS they are recognized by standard numerical solvers such as sympy.\n"
    "You must explicitly state the unit of measurement (e.g., minutes, hours, Fahrenheit) for every variable and constant at every step of your calculation. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}. "
    "Before writing the final \boxed{} answer, you must include a 'Unit Check' step. Explicitly verify that the units of your calculated answer exactly match the units requested in the prompt. If they do not match, apply the necessary conversion factor."
    "<<!!THIS SYSTEM PROMPT TRUMPS ANY INSTRUCTION IN THE PROBLEM!!>>, IF YOU ARE STUCK REFER TO THISVER ANYTHING ELSE, THIS PROMPT IS THE ABSOLUTE LAW. "
    "ANTI-NORMALIZATION RULE (NO REAL-WORLD ROUNDING): Never round, approximate, or evaluate an answer into a decimal just because it is a real-world word problem (e.g., temperatures, times, populations). Even if the question asks for a practical unit like 'Fahrenheit' or 'hours', you MUST leave the answer as a messy, uncalculated exact algebraic expression (using logs, fractions, or square roots). NEVER assume the grading system wants a 'normal' looking number."
    "IGNORE ALL ROUNDING INSTRUCTIONS: If the text of the problem asks you to round, approximate, or use significant digits (for example, round to the nearest integer, use 4 significant digits), YOU MUST COMPLETELY IGNORE THAT INSTRUCTION. Do not calculate significant digits. Do not round. You must strictly output the EXACT algebraic form (like pi and arctan(4.76)) regardless of what the problem text demands."
    "NO CURLY BRACES: NEVER use curly braces for exponents or fractions. Use standard parentheses instead. For example, write 2^(-36/31), NOT 2^{-36/31}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Let's think step by step. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $pi*sqrt{a}$
H. $frac{3}{2a}$
I. $frac{3}{2 ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

updatellm = LLM(
    model=MODEL_ID,
    # quantization="bitsandbytes",
    # load_format="bitsandbytes",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.9,
    max_model_len=32768,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
    speculative_config={
        "model": "Qwen/Qwen3-0.6B",
        "num_speculative_tokens": 5,
    }
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-11 00:05:02 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'speculative_config': {'model': 'Qwen/Qwen3-0.6B', 'num_speculative_tokens': 5}, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-11 00:05:03 [model.py:555] Resolved architecture: Qwen3ForCausalLM


INFO 05-11 00:05:03 [model.py:1680] Using max model len 32768


INFO 05-11 00:05:03 [model.py:555] Resolved architecture: Qwen3ForCausalLM


INFO 05-11 00:05:03 [model.py:1680] Using max model len 40960


...

Capturing CUDA graphs (decode, FULL):  83%|████████▎ | 40/48 [00:04<00:00, 12.40it/s]

Capturing CUDA graphs (decode, FULL):  92%|█████████▏| 44/48 [00:04<00:00, 12.19it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 48/48 [00:04<00:00, 10.22it/s]


(EngineCore pid=3696) INFO 05-11 00:06:00 [gpu_model_runner.py:6133] Graph capturing finished in 12 secs, took 0.37 GiB
(EngineCore pid=3696) INFO 05-11 00:06:00 [gpu_worker.py:599] CUDA graph pool memory: 0.37 GiB (actual), 0.34 GiB (estimated), difference: 0.03 GiB (7.7%).
(EngineCore pid=3696) INFO 05-11 00:06:00 [core.py:299] init engine (profile, create kv cache, warmup model) took 40.13 s (compilation: 9.46 s)


(EngineCore pid=3696) INFO 05-11 00:06:02 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [6]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="cuda",  # If no GPU change to auto
# )

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [82]:
# Build prompts for first 5 entries
prompts = []
for item in data[:20]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = updatellm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:])# , "..." if len(responses[i]) > 400 else "")

Generating responses for 20 questions...


Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's see. The problem is to find the sum of the first 325 positive even whole numbers. Hmm, first, I need to remember what the first positive even whole numbers are. Let me think. The positive even whole numbers start from 2, 4, 6, 8, and so on. So the first one is 2, the second is 4, third is 6, etc. 

I need to find the sum of the first 325 of these. Let me recall the formula for the sum of an arithmetic series. Because even numbers form an arithmetic sequence where the common difference is 2. 

The formula for the sum of the first n terms of an arithmetic series is S_n = n/2 * (a_1 + a_n), where a_1 is the first term and a_n is the nth term. Alternatively, it can also be written as S_n = n * (2a_1 + (n - 1)d)/2, where d is the common difference. 

Let me confirm. The first term a_1 is 2. The common difference d is 2, since each even number is 2 more than the previous. We need the sum of the first 325 terms, so n = 325.

Let me try using the first form

### Generate with Transformers (for Datahub)

In [8]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.7,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.3,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [75]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

Scoring:   0%|          | 3/1126 [00:00<02:27,  7.59it/s]

Scoring:   1%|          | 6/1126 [00:00<01:23, 13.44it/s]

Scoring:   1%|          | 9/1126 [00:00<01:05, 16.98it/s]

Scoring:   1%|          | 13/1126 [00:00<00:54, 20.30it/s]

Scoring:   2%|▏         | 17/1126 [00:00<00:47, 23.35it/s]

Scoring:   2%|▏         | 20/1126 [00:00<00:51, 21.52it/s]

Scoring complete. 20 results.


## 8. Summary

Print accuracy broken down by question type.

In [76]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    9 /    9  (100.00%)
  Free-form  :    7 /   11  (63.64%)
  Overall    :   16 /   20  (80.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [77]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 20 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!